# Reference: Fairshake Cloud Fairness Metrics
For more information on fairness metrics and evaluation, see the [Fairshake Cloud documentation](https://fairshake.cloud/documentation/).


In [ ]:
import os
# Set working directory to your Synthea project folder
os.chdir("/Users/jay/Documents/ICA/New/synthea-master")
print("📂 Current working directory:", os.getcwd())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load data
patients = pd.read_csv('/Users/jay/Documents/ICA/New/synthea-master/output/merged_csv/patients.csv')
conditions = pd.read_csv('/Users/jay/Documents/ICA/New/synthea-master/output/merged_csv/conditions.csv')
encounters = pd.read_csv("/Users/jay/Documents/ICA/New/synthea-master/output/merged_csv/encounters.csv")

In [ ]:
# Check the unique values in STATE column
unique_states = patients['STATE'].dropna().unique()

In [ ]:
print(f"✅ Found {len(unique_states)} unique STATE values:")
print(sorted(unique_states))

In [ ]:
# Mental health-related SNOMED root codes
mental_health_code_roots = [
    "74732009",    # Mental disorder
    "35489007",    # Depressive disorder
    "197480006",   # Anxiety disorder
    "13746004",    # Bipolar disorder
    "394687007",   # Suicide-related behavior
    "58214004"     # Schizophrenia
]

In [ ]:
# Filter by any code that starts with these
mental_health_conditions = conditions[conditions['CODE'].astype(str).isin(mental_health_code_roots)]

In [ ]:
print(f"🧠 Mental health-related rows: {len(mental_health_conditions)}")
print("🔍 Example diagnoses:")
print(mental_health_conditions['DESCRIPTION'].value_counts().head())

In [ ]:
# Define mental health keywords
mh_keywords = [
    'depression', 'depressive', 'dysthymia', 'mood disorder', 'major depressive',
    'anxiety', 'panic', 'phobia', 'agoraphobia', 'generalized anxiety', 'gad',
    'ptsd', 'post-traumatic stress', 'bipolar', 'manic', 'mania',
    'ocd', 'obsessive compulsive', 'obsessions', 'compulsions',
    'adhd', 'attention deficit', 'hyperactivity',
    'schizophrenia', 'psychosis', 'psychotic', 'hallucination', 'delusion',
    'suicide', 'suicidal', 'self-harm', 'self injury',
    'substance', 'alcohol', 'drug abuse', 'opioid', 'stimulant', 'addiction', 'dependence',
    'eating disorder', 'anorexia', 'bulimia', 'binge eating',
    'personality disorder', 'borderline', 'antisocial', 'paranoid',
    'mental', 'mental illness', 'emotional disturbance', 'behavioral disorder'
]

In [ ]:
# Filter conditions for mental health
mh_conditions = conditions[
    conditions['DESCRIPTION'].str.lower().str.contains('|'.join(mh_keywords))
]

In [ ]:
# Patients with mental health conditions
mh_patient_ids = mh_conditions['PATIENT'].unique()
mh_patients = patients[patients['Id'].isin(mh_patient_ids)]
mh_patients.head()

In [ ]:
# === Summary Stats ===
print("🧠 Mental Health EDA Summary")
print(f"Total patients: {len(patients):,}")
print(f"Patients with mental health conditions: {len(mh_patients):,}")
print(f"Percent with mental health issues: {100 * len(mh_patients) / len(patients):.2f}%\n")

In [ ]:
# === Top Conditions ===
top_conditions = mh_conditions['DESCRIPTION'].value_counts().head(10)
print("Top Mental Health Conditions:")
print(top_conditions)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import numpy as np

# Suppress all warnings
warnings.filterwarnings('ignore')

# Clean copy and calculate age
mh_patients = mh_patients.copy()
mh_patients.loc[:, 'BIRTHDATE'] = pd.to_datetime(mh_patients['BIRTHDATE'], errors='coerce')
today = pd.to_datetime('today')
mh_patients.loc[:, 'AGE'] = (today - mh_patients['BIRTHDATE']).dt.days // 365

# Create bins and viridis color gradient
bins = np.linspace(mh_patients['AGE'].min(), mh_patients['AGE'].max(), 21)
colors = sns.color_palette("viridis", len(bins) - 1)

# Plot
plt.figure(figsize=(10, 6))
n, bins, patches = plt.hist(mh_patients['AGE'], bins=bins, edgecolor='black')

# Apply gradient colors
for patch, color in zip(patches, colors):
    patch.set_facecolor(color)

# Set ticks every 10 years
min_age = int(mh_patients['AGE'].min())
max_age = int(mh_patients['AGE'].max())
plt.xticks(np.arange(min_age, max_age + 1, 10))

# Labels and title
plt.title("Age Distribution of Mental Health Patients", fontsize=14)
plt.xlabel("Age (years)", fontsize=12)
plt.ylabel("Patient Count", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

# Convert birthdate and calculate age
mh_patients['BIRTHDATE'] = pd.to_datetime(mh_patients['BIRTHDATE'], errors='coerce')
today = pd.to_datetime('today')
mh_patients['AGE'] = (today - mh_patients['BIRTHDATE']).dt.days // 365
mh_patients = mh_patients[mh_patients['AGE'].notnull()]

# 🎨 Vibrant, aesthetic colors
palette = {
    'M': '#007aff',  
    'F': '#e63946'  
}

# Plot
plt.figure(figsize=(12, 6))
sns.set(style="white")

hist = sns.histplot(
    data=mh_patients,
    x='AGE',
    bins=30,
    hue='GENDER',
    multiple='stack',
    palette=palette,
    edgecolor='black'
)

# Style
plt.title("Age Distribution of Mental Health Patients by Gender", fontsize=15, weight='bold')
plt.xlabel("Age", fontsize=12)
plt.ylabel("Number of Patients", fontsize=12)
plt.xticks(range(0, 101, 10))
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# === Encounter Types (optional) ===
mh_encounters = encounters[encounters['PATIENT'].isin(mh_patient_ids)]
encounter_counts = mh_encounters['ENCOUNTERCLASS'].value_counts()
print("\n🏥 Encounter Types for Mental Health Patients:")
print(encounter_counts)

In [ ]:
under_21_count = mh_patients[mh_patients['AGE'] < 21].shape[0]
print(f"Mental health patients under 21: {under_21_count}")

total = mh_patients.shape[0]
under_21_pct = (under_21_count / total) * 100
print(f"Under 21: {under_21_count} patients ({under_21_pct:.2f}%)")

In [ ]:
# Count patients by county
county_counts = mh_patients['COUNTY'].value_counts().reset_index()
county_counts.columns = ['County', 'Patient Count']

# Plot
plt.figure(figsize=(12, 6))
sns.barplot(data=county_counts.head(20), x='Patient Count', y='County', palette='Set1')

# Style
plt.title("Top 20 Counties by Mental Health Patient Count", fontsize=14, weight='bold')
plt.xlabel("Number of Patients")
plt.ylabel("County")
plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px
import pandas as pd

# === Prepare your data ===
# mh_patients should already be filtered
# Ensure you have FIPS codes for each patient’s county
# For now, aggregate patient counts by COUNTY + STATE
mh_by_county = mh_patients.groupby(['STATE', 'COUNTY']).size().reset_index(name='count')

# Load a FIPS crosswalk file to get county-level FIPS codes
# Download from: https://www2.census.gov/programs-surveys/popest/geographies/2016/all-geocodes-v2016.csv
# Or use a prebuilt one (sample included below)
fips_ref = pd.read_csv("https://raw.githubusercontent.com/plotly/datasets/master/fips-unemp-16.csv")
fips_ref = fips_ref[['fips', 'County']]

# Merge FIPS codes into your mental health data
mh_by_county['County'] = mh_by_county['COUNTY'].str.title()
plot_data = mh_by_county.merge(fips_ref, on='County', how='left')

# === Plotly Choropleth ===
fig = px.choropleth(
    plot_data,
    geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
    locations='fips',
    color='count',
    color_continuous_scale="Reds",
    range_color=(0, plot_data['count'].max()),
    scope="usa",
    labels={'count': 'Mental Health Patients'},
    title="🧠 Mental Health Patients by U.S. County"
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()